# [9665]  Exercise : Sentiment Analysis - Solution
Data file:
* https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/IMDB_movie_reviews_5k.csv

## Exercise Requirements
* Load data into dataframe
* Perform appropriate text preprocessing & text vectorization
* Choose a learning algorithm to use for model training
* Prepare data for model training
* Train model
* Display model accuracy

In [ ]:
from datetime import datetime
print(f'Run time: {datetime.now().strftime("%D %T")}')

Run time: 02/09/25 13:34:56


### Import libraries

In [ ]:
import pandas as pd
import re
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score

In [ ]:
nltk.download('wordnet')
nltk.download('stopwords')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

### Load data
* Independent variable: review
* Dependent variable: sentiment

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/IMDB_movie_reviews_5k.csv')

### Examine data

In [ ]:
pd.set_option('max_colwidth', None)

In [ ]:
df.shape

(5000, 2)

In [ ]:
# Display first few rows of updated dataframe
df.sample(2)

,review,sentiment
3396,"Duchess and her three kittens are enjoying the high life with their devoted human mistress until the wicked butler Edgar, with his eyes on a big inheritance, decides to dope them and get them out of the picture. How can these fragile creatures cope in the unfamiliar countryside and the meaner streets of Paris? Only by meeting the irrepressible alley cat O'Malley, a rough diamond with romance in his heart. After they get a taste of the wide dangerous world, he guides them home, and Edgar gets his just desserts at the wrong end of a horse. As always, it's really the voices rather than the animation that are the heart of the Disney magic: Phil Harris is brilliant as O'Malley, Eva Gabor as Duchess is... well... Eva Gabor; but perhaps the most memorable turns are by Pat Buttram and George Lindsay, who turn the old hounds Napoleon and Lafayette into a couple of bumbling Southern-fried rednecks. Their scenes with Edgar, and the musical numbers with Scat Cat and his cool-dude band, are classic. Most striking about seeing The Aristocats now is how deeply Disney's style of animation has changed since this was at the cutting edge in 1970. Perhaps the nostalgic, dated feel are just a result of being plonked down in Belle Epoque Paris, but the illustrations are fussier (a pity) and the animation and overall pace much less frenetic (sometimes a relief) than in more recent efforts such as Aladdin.",1
3873,"Between the ages of 30 and 51, when he died of a brain tumour, Zachary Scott made 70 films. He was introduced in 1944 in Jean Negulesco's 'The Mask of Dimitrios', where he played Dimitrios. The next year, 1945, he made three films, of which this is one. He is best remembered by cineastes as the star of Jean Renoir's 'The Southerner', one of the 1945 films, where he had a sympathetic role. However, he often played creepy characters, and in this film he is a sociopathic killer of women for money. So what happens here? He lives in a house with three women, so watch out! Faye Emerson, who also appeared in 'Dimitrios', plays the older of two daughters in the house. She falls in love with Scott and they become secretly engaged. Then her 'cute kid' younger sister (played effectively by Mona Freeman, who resembles Bonita Granville both in looks and in behaviour) returns from boarding school and reveals casually in conversation with Scott that she has inherited a tidy sum, so Scott turns his sights on her instead, with all the torrid jealousies seething in the household which that was bound to arouse. Things get tense, and then they get tenser. Meanwhile, plans for murder are going forward in the mind of the calculating Scott. But it turns out that he is not the only one with such intentions. He is also being searched for as a result of his last kill, with which the film has opened, so that we know his back story. James Wong Howe gives effective noirish cinematography to this tale, which was directed by Frenchman Robert Florey who had moved to Hollywood some time earlier. The film is an effective psychopath-in-the-house mystery which can cause a bit of wear of the edges of some seats, for those of such an inclination.",1


### Clean data

In [ ]:
# Remove the string '<br /><br />' from 'review' column
df['review'] = df['review'].str.replace('<br /><br />', ' ')

In [ ]:
# Remove extra spaces from 'review' column
df['review'] = df['review'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [ ]:
# Display first few rows of updated dataframe
df.head(2)

,review,sentiment
0,"Inept, boring, and incoherent supernatural ""thriller"" in which college student Cassie (Melissa Sagemiller) is the constant victim of hallucinations and nightmares after a car accident claims the life of her boyfriend Sean (Casey Affleck). I can't begin to tell you how bad this is...nothing of any importance ever happens nor is there ever any sort of actual entertainment value. I did not like this cast in this particular film - they are all sadly unconvincing (then again, their roles are no good). To promote this as a horror film is a joke. Where are the scares? There's no sense or suspense - there are a few good songs but that's about it. How on Earth did this project get the green light? Writer-director Steve Carpenter has no discernible vision or talent that I can sense. Worst of all, the conclusion really makes the whole movie pointless. The alleged ""killer cut"" that I watched is 86 minutes of pure tedium. 1/10",0
1,"Jimmy Cagney races by your eyes constantly in this story of a stage-producer who is vigorously struggling against the upcoming ""talking"" movies. This story of love, deceit, women and dancing is presented in such a manner that as a viewer you are never treated to a dull moment. The direction of the mass scenes in the rehearsal rooms was enormously well done. The story never really got lost in this frantic pace. Some parts of the material presented here have become a little dated but that doesn't matter because when you look at this in a 1933 time-frame it is fabulous to watch this next to a lot of the other drags of movies that were released during that time. Jimmy Cagney is a sight for sore eyes in this film, never loosing his composure as the ever-working producer of previews made for the movie theaters as intros. In this way he tries to save his ass from going out of business, he was a broadway producer before he started this. Joan Blondell is fabulous as the neglected love-interest, Nan, she gives such a spirited performance that is so unusual for movies of that time, so cool to watch a woman who is portrayed as a strong woman for a change. The only problem I had with the film were the enormous productions at the end. These were magnificent in itself, beautifully choreographed and wonderfully produced, but they just didn't seem to fit in the story. The only link they have to the main story is that Cagney had to put on 3 previews in 3 days to get a contract and that's what he did. I had a hard time believing that this was what the girls had been rehearsing during the entire movie and that these sets could fit in a movie theater. In this way the ""Sitting On A Backyard Fence"" was much more appropriate to the story. The productions at the end seemed to drag this frantically paced story to a halt and that was not a good thing. I was tired after seeing the first Musical sequence and then I realized there were another two coming up. These sequences got a lot a chuckles from the audience as well. All in all a great film with a sour ending. 9/10",1


### Prepare data

#### Create function to lowercase, remove punctuation, tokenize, remove stopwords, and lemmatize

In [ ]:
# Define stopwords
stopwords = nltk.corpus.stopwords.words('english')

In [ ]:
# Instantiate lemmatizer
lem = WordNetLemmatizer()

In [ ]:
# Function clean_text will be used in subsequent cells
def clean_text(text):
    text = "".join([word.lower() for word in text if word not in string.punctuation])
    tokens = re.split('\W+', text)
    text = [lem.lemmatize(word) for word in tokens if word not in stopwords]
    text_2 = ' '.join(word for word in text)
    return text_2     # Returns one string

In [ ]:
%%time

# Apply function to clean 'review' column
df['review_clean'] = df['review'].apply(clean_text)

CPU times: user 13.5 s, sys: 356 ms, total: 13.9 s
Wall time: 24.1 s


In [ ]:
# Display first few rows of updated dataframe
df.head(2)

,review,sentiment,review_clean
0,"Inept, boring, and incoherent supernatural ""thriller"" in which college student Cassie (Melissa Sagemiller) is the constant victim of hallucinations and nightmares after a car accident claims the life of her boyfriend Sean (Casey Affleck). I can't begin to tell you how bad this is...nothing of any importance ever happens nor is there ever any sort of actual entertainment value. I did not like this cast in this particular film - they are all sadly unconvincing (then again, their roles are no good). To promote this as a horror film is a joke. Where are the scares? There's no sense or suspense - there are a few good songs but that's about it. How on Earth did this project get the green light? Writer-director Steve Carpenter has no discernible vision or talent that I can sense. Worst of all, the conclusion really makes the whole movie pointless. The alleged ""killer cut"" that I watched is 86 minutes of pure tedium. 1/10",0,inept boring incoherent supernatural thriller college student cassie melissa sagemiller constant victim hallucination nightmare car accident claim life boyfriend sean casey affleck cant begin tell bad isnothing importance ever happens ever sort actual entertainment value like cast particular film sadly unconvincing role good promote horror film joke scare there sense suspense good song thats earth project get green light writerdirector steve carpenter discernible vision talent sense worst conclusion really make whole movie pointless alleged killer cut watched 86 minute pure tedium 110
1,"Jimmy Cagney races by your eyes constantly in this story of a stage-producer who is vigorously struggling against the upcoming ""talking"" movies. This story of love, deceit, women and dancing is presented in such a manner that as a viewer you are never treated to a dull moment. The direction of the mass scenes in the rehearsal rooms was enormously well done. The story never really got lost in this frantic pace. Some parts of the material presented here have become a little dated but that doesn't matter because when you look at this in a 1933 time-frame it is fabulous to watch this next to a lot of the other drags of movies that were released during that time. Jimmy Cagney is a sight for sore eyes in this film, never loosing his composure as the ever-working producer of previews made for the movie theaters as intros. In this way he tries to save his ass from going out of business, he was a broadway producer before he started this. Joan Blondell is fabulous as the neglected love-interest, Nan, she gives such a spirited performance that is so unusual for movies of that time, so cool to watch a woman who is portrayed as a strong woman for a change. The only problem I had with the film were the enormous productions at the end. These were magnificent in itself, beautifully choreographed and wonderfully produced, but they just didn't seem to fit in the story. The only link they have to the main story is that Cagney had to put on 3 previews in 3 days to get a contract and that's what he did. I had a hard time believing that this was what the girls had been rehearsing during the entire movie and that these sets could fit in a movie theater. In this way the ""Sitting On A Backyard Fence"" was much more appropriate to the story. The productions at the end seemed to drag this frantically paced story to a halt and that was not a good thing. I was tired after seeing the first Musical sequence and then I realized there were another two coming up. These sequences got a lot a chuckles from the audience as well. All in all a great film with a sour ending. 9/10",1,jimmy cagney race eye constantly story stageproducer vigorously struggling upcoming talking movie story love deceit woman dancing presented manner viewer never treated dull moment direction mass scene rehearsal room enormously well done story never really got lost frantic pace part material presented become little dated doesnt matter look 1933 timefra

### Separate independent and dependent variables

In [ ]:
X = df[['review_clean']]
y = df['sentiment']

### Vectorize independent variables using TF-IDF vectorizer

In [ ]:
%%time

# Word (unigram) technique - Create feature vectors using Bag of Words-TfIdf
wtfidf_converter = TfidfVectorizer(max_features=10000, min_df=5, max_df=0.7)
X_word = wtfidf_converter.fit_transform(X['review_clean']).toarray()
X_word.shape

CPU times: user 1.11 s, sys: 448 ms, total: 1.56 s
Wall time: 1.64 s


(5000, 10000)

In [ ]:
%%time

# Bigrams - Create feature vectors using Ngrams-TfIdf
ntfidf_converter = TfidfVectorizer(max_features=10000, min_df=5, max_df=0.7, ngram_range=(2,2))
X_ngram = ntfidf_converter.fit_transform(X['review_clean']).toarray()
X_ngram.shape

CPU times: user 2.44 s, sys: 275 ms, total: 2.71 s
Wall time: 2.71 s


(5000, 8629)

### Split data into training and test sets

In [ ]:
# Split unigram data into training and test sets
X_train_1, X_test_1, y_train_1, y_test_1 = \
    train_test_split(X_word, y, test_size=.25, random_state=42)

In [ ]:
# Split bigram data into training and test sets
X_train_2, X_test_2, y_train_2, y_test_2 = \
    train_test_split(X_ngram, y, test_size=.25, random_state=42)

### Train random forest models

In [ ]:
%%time

rf_word = RandomForestClassifier(random_state=42)
rf_word.fit(X_train_1, y_train_1)

CPU times: user 12.1 s, sys: 83.2 ms, total: 12.1 s
Wall time: 12.2 s


RandomForestClassifier(random_state=42)

In [ ]:
%%time

rf_ngram = RandomForestClassifier(random_state=42)
rf_ngram.fit(X_train_2, y_train_2)

CPU times: user 50.7 s, sys: 132 ms, total: 50.8 s
Wall time: 56.4 s


RandomForestClassifier(random_state=42)

### Evaluate random forest models

In [ ]:
y_pred_1 = rf_word.predict(X_test_1)
rf_accuracy_1 = accuracy_score(y_test_1, y_pred_1)
print('Random forest (word + tfidf) Accuracy:', round((rf_accuracy_1 * 100), 5), "%")

Random forest (word + tfidf) Accuracy: 82.4 %


In [ ]:
y_pred_2 = rf_ngram.predict(X_test_2)
rf_accuracy_2 = accuracy_score(y_test_2, y_pred_2)
print('Random forest (ngram + tfidf) Accuracy:', round((rf_accuracy_2 * 100), 5), "%")

Random forest (ngram + tfidf) Accuracy: 75.76 %


### Train AdaBoost models

In [ ]:
%%time

ab_word = AdaBoostClassifier(random_state=42)
ab_word.fit(X_train_1, y_train_1)

CPU times: user 34.2 s, sys: 3.01 s, total: 37.3 s
Wall time: 39 s


AdaBoostClassifier(random_state=42)

In [ ]:
%%time

ab_ngram = AdaBoostClassifier(random_state=42)
ab_ngram.fit(X_train_2, y_train_2)

CPU times: user 26.5 s, sys: 2.54 s, total: 29.1 s
Wall time: 31.8 s


AdaBoostClassifier(random_state=42)

### Evaluate AdaBoost models

In [ ]:
y_pred_1 = ab_word.predict(X_test_1)
ab_accuracy_1 = accuracy_score(y_test_1, y_pred_1)
print('AdaBoost (word + tfidf) Accuracy:', round((ab_accuracy_1 * 100), 5), "%")

AdaBoost (word + tfidf) Accuracy: 76.0 %


In [ ]:
y_pred_2 = ab_ngram.predict(X_test_2)
ab_accuracy_2 = accuracy_score(y_test_2, y_pred_2)
print('AdaBoost (ngram + tfidf) Accuracy:', round((ab_accuracy_2 * 100), 5), "%")

AdaBoost (ngram + tfidf) Accuracy: 60.88 %
